# 有严重自杀意念青少年中识别同期自杀未遂：模型构建与评估（YRBS 2019–2025，18 个变量）

**数据**：与本 notebook 放在同一文件夹的 `yrbs_ideators_imputed_full.csv`（四年合并、已插补，9,824 人）。
代码按"结局 × 年份"分层随机划分 7:3（random_state = 20）。缺失值插补模型只在这 70% 训练集上拟合，再应用到 30% 测试集，因此不要更改种子或划分方式，否则插补与变量筛选将不再只基于训练集。
18 个预测变量由训练集上 LASSO（λ1se）与 Boruta 同时入选确定（候选变量含"过去 12 个月有自杀计划"）。

**流程**：固定超参数（已在训练集上用分层 10 折交叉验证网格搜索、以 AUROC 选出）→ 训练集拟合 8 个模型 →
训练集 10 折交叉验证预测确定 F1 最优阈值 → 测试集评估（正文 Table 2、ROC、校准）→ 训练集交叉验证表现（附表 Table S、附图）→ DeLong 检验 → 不同患病率下 PPV/NPV → DCA → SHAP。

**首次运行前**（Anaconda Prompt 中执行一次）：

```
pip install scikit-learn xgboost lightgbm shap statsmodels openpyxl matplotlib
```

全部结果保存在 `results/` 文件夹。完整运行约 10–15 分钟（SVM 的概率输出较慢）。

In [ ]:
# ===== 0. 设置 =====
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, roc_curve, average_precision_score, brier_score_loss,
                             confusion_matrix, f1_score)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")
SEED = 2026              # 模型与交叉验证的随机种子
SPLIT_SEED = 20          # 7:3 划分的随机种子（插补、变量筛选、调参均基于该划分的训练集，请勿更改）
DATA_DIR = os.path.join("data", "derived")   # output of 04_split_impute.py
OUT_DIR = "results"
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300})
PAPER_RC = {"font.family": "serif", "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"],
            "mathtext.fontset": "stix", "axes.unicode_minus": False}
print("scikit-learn", sklearn.__version__)

In [ ]:
# ===== 1. 读取数据与变量 =====
from sklearn.model_selection import train_test_split
full = pd.read_csv(os.path.join(DATA_DIR, "yrbs_ideators_imputed_full.csv"))
strata = full["sa"].astype(int).astype(str) + "_" + full["year"].astype(str)        # 按结局 × 年份分层
train, test = train_test_split(full, test_size=0.30, stratify=strata, random_state=SPLIT_SEED)
# 按年份、编号排序，保证交叉验证分折和随机过程可完全复现
train = train.sort_values(["year", "record"]).reset_index(drop=True)
test = test.sort_values(["year", "record"]).reset_index(drop=True)
train.to_csv(os.path.join(OUT_DIR, "train_70.csv"), index=False, encoding="utf-8-sig")
test.to_csv(os.path.join(OUT_DIR, "test_30.csv"), index=False, encoding="utf-8-sig")

VARS = ['grade', 'female', 'raceeth', 'sexual_minority', 'sad_hopeless', 'plan', 'nicotine_status', 'alcohol_status', 'marijuana_status', 'other_drug_any', 'early_onset_any', 'bullying_count', 'school_victim_any', 'fighting_any', 'weapon_any', 'sexual_dating_violence_any', 'sexual_behavior', 'concussion']
CAT_LEVELS = {"raceeth": ["White", "Asian", "Black", "Hispanic", "Other"],   # 第一个水平为参照组
              "nicotine_status": [0, 1, 2], "alcohol_status": [0, 1, 2],
              "marijuana_status": [0, 1, 2], "sexual_behavior": [0, 1, 2]}
CAT = [c for c in CAT_LEVELS if c in VARS]
NUM = [v for v in VARS if v not in CAT]
LABELS = {"grade": "Grade", "female": "Female sex", "sexual_minority": "Sexual minority",
          "sad_hopeless": "Persistent sadness/hopelessness", "plan": "Suicide plan", "sleep_hours": "Sleep hours",
          "age": "Age", "bmi": "BMI", "fv_none_count": "Fruit/vegetable categories not eaten", "no_breakfast": "No breakfast",
          "drank_soda": "Soda consumption", "pa_5days": "Physically active >=5 d/wk", "sports_pe_any": "Sports team or PE",
          "other_drug_any": "Lifetime illicit drug use or prescription opioid misuse",
          "early_onset_any": "Substance use before 13", "bullying_count": "Bullying victimization",
          "school_victim_any": "School violence/unsafe", "fighting_any": "Physical fighting",
          "weapon_any": "Weapon carrying", "sexual_dating_violence_any": "Sexual/dating violence",
          "concussion": "Sports concussion", "dentist": "Dental visit",
          "hiv_std_test_any": "HIV/STD testing",
          "raceeth_Asian": "Race: Asian (vs White)", "raceeth_Black": "Race: Black (vs White)",
          "raceeth_Hispanic": "Race: Hispanic (vs White)", "raceeth_Other": "Race: Other (vs White)",
          "nicotine_status_1": "Tobacco/nicotine: former", "nicotine_status_2": "Tobacco/nicotine: current",
          "alcohol_status_1": "Alcohol: former", "alcohol_status_2": "Alcohol: current",
          "marijuana_status_1": "Marijuana: former", "marijuana_status_2": "Marijuana: current",
          "sexual_behavior_1": "Sexual behavior: ever", "sexual_behavior_2": "Sexual behavior: high-risk"}

X_train, y_train = train[VARS], train["sa"].astype(int).values
X_test, y_test = test[VARS], test["sa"].astype(int).values
print(f"全部 n={len(full)}, 未遂 {int(full.sa.sum())} ({full.sa.mean():.1%})")
print(f"训练集 n={len(train)}, 未遂 {y_train.sum()} ({y_train.mean():.1%})")
print(f"测试集 n={len(test)}, 未遂 {y_test.sum()} ({y_test.mean():.1%})")
display(pd.crosstab(pd.concat([train.year, test.year], keys=["train", "test"]).reset_index(level=0)["level_0"],
                    pd.concat([train.year, test.year]).values, rownames=["set"], colnames=["year"]))

In [ ]:
# ===== 2. 固定超参数（训练集分层 10 折交叉验证网格搜索、AUROC 选优得到）=====
BEST_PARAMS = {
    "LR": {"C": 0.1, "penalty": 'l2'},
    "RF": {"max_depth": 8, "min_samples_leaf": 10},
    "SVM": {"C": 30, "gamma": 0.001},
    "XGB": {"learning_rate": 0.05, "max_depth": 2, "n_estimators": 300},
    "LightGBM": {"n_estimators": 300, "num_leaves": 3},
    "MLP": {"alpha": 3, "hidden_layer_sizes": (32,)},
    "KNN": {"n_neighbors": 300, "weights": 'uniform'},
    "NB": {"var_smoothing": 20},
}

def make_preprocessor():
    kw = {"sparse_output": False} if tuple(int(x) for x in sklearn.__version__.split(".")[:2]) >= (1, 2) else {"sparse": False}
    ohe = OneHotEncoder(categories=[CAT_LEVELS[c] for c in CAT], drop="first", handle_unknown="ignore", **kw)
    return ColumnTransformer([("num", StandardScaler(), NUM), ("cat", ohe, CAT)])

def make_model(name):
    p = BEST_PARAMS[name]
    est = {
        "LR": lambda: LogisticRegression(solver="liblinear", max_iter=5000, **p),
        "RF": lambda: RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1, **p),
        "SVM": lambda: SVC(kernel="rbf", probability=True, random_state=SEED, **p),
        "XGB": lambda: XGBClassifier(subsample=0.8, colsample_bytree=0.8, random_state=SEED, n_jobs=-1,
                                     eval_metric="logloss", **p),
        "LightGBM": lambda: LGBMClassifier(learning_rate=0.05, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                                           random_state=SEED, n_jobs=-1, verbose=-1, **p),
        "MLP": lambda: MLPClassifier(max_iter=1000, early_stopping=True, random_state=SEED, **p),
        "KNN": lambda: KNeighborsClassifier(**p),
        "NB": lambda: GaussianNB(**p),
    }[name]()
    return Pipeline([("prep", make_preprocessor()), ("clf", est)])

MODEL_NAMES = ["LR", "RF", "SVM", "XGB", "LightGBM", "MLP", "KNN", "NB"]
pd.DataFrame([{"Model": m, "Hyperparameters": json.dumps(BEST_PARAMS[m])} for m in MODEL_NAMES])

In [ ]:
# ===== 2b.（可选）重新调参：默认不运行。若要复现调参过程，把 RUN_TUNING 改为 True =====
RUN_TUNING = False
if RUN_TUNING:
    GRIDS = {"LR": {"C": [0.01, 0.1, 1, 10, 100], "penalty": ['l1', 'l2']},
             "RF": {"max_depth": [6, 8, 10, None], "min_samples_leaf": [1, 2, 5, 10, 20]},
             "SVM": {"C": [0.1, 1, 3, 10, 30, 100], "gamma": [0.0003, 0.001, 0.003]},
             "XGB": {"n_estimators": [100, 300], "max_depth": [2, 4], "learning_rate": [0.05, 0.1]},
             "LightGBM": {"num_leaves": [2, 3, 7], "n_estimators": [100, 300, 800, 1500]},
             "MLP": {"hidden_layer_sizes": [(32,), (64, 32)], "alpha": [0.01, 0.1, 1, 3, 10]},
             "KNN": {"n_neighbors": [100, 150, 200, 300, 400, 600], "weights": ['uniform']},
             "NB": {"var_smoothing": [0.1, 0.3, 1, 2, 5, 10, 20]}}
    cv = StratifiedKFold(10, shuffle=True, random_state=SEED)
    for m in MODEL_NAMES:
        saved = BEST_PARAMS[m]; BEST_PARAMS[m] = {}
        pipe = make_model(m)
        if m == "SVM":
            pipe.set_params(clf__probability=False)
        gs = GridSearchCV(pipe, {f"clf__{k}": v for k, v in GRIDS[m].items()}, scoring="roc_auc", cv=cv, n_jobs=-1)
        gs.fit(X_train, y_train)
        BEST_PARAMS[m] = {k.replace("clf__", ""): v for k, v in gs.best_params_.items()}
        print(m, BEST_PARAMS[m], round(gs.best_score_, 4))

In [ ]:
# ===== 3. 拟合模型；训练集 10 折交叉验证预测（用于确定 F1 最优阈值）；测试集预测 =====
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
models, p_train_cv, p_test = {}, {}, {}
for m in MODEL_NAMES:
    p_train_cv[m] = cross_val_predict(make_model(m), X_train, y_train, cv=cv10, method="predict_proba")[:, 1]
    models[m] = make_model(m).fit(X_train, y_train)
    p_test[m] = models[m].predict_proba(X_test)[:, 1]
    print(f"{m:9s} 训练集CV AUC={roc_auc_score(y_train, p_train_cv[m]):.3f}   测试集 AUC={roc_auc_score(y_test, p_test[m]):.3f}")
pd.DataFrame(p_test).assign(sa=y_test, year=test["year"].values, record=test["record"].values) \
  .to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False)

In [ ]:
# ===== 4. 评价指标函数：DeLong、Bootstrap、阈值指标、Hosmer–Lemeshow =====
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(N); out[J] = T
    return out

def delong(y, preds):
    """Fast DeLong (Sun & Xu 2014). preds: (k, n). Returns AUCs and covariance matrix."""
    y = np.asarray(y); order = np.argsort(-y); y = y[order]; preds = np.atleast_2d(preds)[:, order]
    m = int(y.sum()); n = len(y) - m
    pos, neg = preds[:, :m], preds[:, m:]
    k = preds.shape[0]
    tx = np.array([_midrank(r) for r in pos]); ty = np.array([_midrank(r) for r in neg])
    tz = np.array([_midrank(r) for r in preds])
    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    cov = np.cov(v01) / m + np.cov(v10) / n
    return aucs, np.atleast_2d(cov)

def auc_ci_delong(y, p, alpha=0.05):
    auc, cov = delong(y, p[None, :])
    se = np.sqrt(cov[0, 0]); z = stats.norm.ppf(1 - alpha / 2)
    return auc[0], max(0, auc[0] - z * se), min(1, auc[0] + z * se)

def delong_test(y, p1, p2):
    aucs, cov = delong(y, np.vstack([p1, p2]))
    var = cov[0, 0] + cov[1, 1] - 2 * cov[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return aucs[0] - aucs[1], 2 * stats.norm.sf(abs(z))

def auprc_ci(y, p, B=1000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); vals = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if y[i].min() != y[i].max():
            vals.append(average_precision_score(y[i], p[i]))
    return average_precision_score(y, p), np.percentile(vals, 2.5), np.percentile(vals, 97.5)

def threshold_metrics(y, p, t):
    pred = (p >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn); spec = tn / (tn + fp)
    ppv = tp / (tp + fp) if tp + fp else np.nan; npv = tn / (tn + fn) if tn + fn else np.nan
    return {"Accuracy": (tp + tn) / len(y), "Recall": sens, "Specificity": spec, "PPV": ppv, "NPV": npv,
            "F1 score": f1_score(y, pred, zero_division=0)}

def f1_optimal_threshold(y, p, grid=np.round(np.arange(0.01, 1.00, 0.01), 2)):
    f1s = [f1_score(y, (p >= t).astype(int), zero_division=0) for t in grid]
    return float(grid[int(np.argmax(f1s))])

def hosmer_lemeshow(y, p, g=10):
    d = pd.DataFrame({"y": y, "p": p})
    d["grp"] = pd.qcut(d["p"].rank(method="first"), g, labels=False)
    s = d.groupby("grp").agg(o=("y", "sum"), e=("p", "sum"), n=("y", "size"))
    hl = (((s.o - s.e) ** 2) / (s.e * (1 - s.e / s.n))).sum()
    return hl, stats.chi2.sf(hl, g - 2)

In [ ]:
# ===== 5. 表现汇总函数；正文 Table 2（测试集）=====
F1_THRESHOLDS = {m: f1_optimal_threshold(y_train, p_train_cv[m]) for m in MODEL_NAMES}   # 只用训练集交叉验证预测确定

def performance_table(y, preds):
    rows = []
    for m in MODEL_NAMES:
        p = preds[m]
        auc, lo, hi = auc_ci_delong(y, p)
        ap, alo, ahi = auprc_ci(y, p)
        hl, hl_p = hosmer_lemeshow(y, p)
        for label, t in [("Default", 0.50), ("F1-optimal", F1_THRESHOLDS[m])]:
            rows.append({"Models": m, "Decision Label": label, "Thresholds": t,
                         "AUC-ROC (95%CI)": f"{auc:.3f} ({lo:.3f}-{hi:.3f})",
                         "AUC-PRC (95%CI)": f"{ap:.3f} ({alo:.3f}-{ahi:.3f})",
                         **{k: ("NA" if pd.isna(v) else round(v, 3)) for k, v in threshold_metrics(y, p, t).items()},
                         "H-L test p": f"{hl_p:.3f}" if hl_p >= 0.001 else "<0.001",
                         "Brier score": round(brier_score_loss(y, p), 3), "_auc": auc})
    return pd.DataFrame(rows)

table2 = performance_table(y_test, p_test)
BEST_MODEL = "LR"   # selected on cross-validated training performance (simplest among comparable models)
table2 = table2.drop(columns="_auc")
table2.to_csv(os.path.join(OUT_DIR, "Table2_test_performance.csv"), index=False, encoding="utf-8-sig")
table2.to_excel(os.path.join(OUT_DIR, "Table2_test_performance.xlsx"), index=False)
print("Table 2 (test set). Final model:", BEST_MODEL)
table2

In [ ]:
# ===== 5b. 附表 Table S：训练集表现（10 折交叉验证预测，格式同 Table 2）=====
tableS = performance_table(y_train, p_train_cv).drop(columns="_auc")
tableS.to_csv(os.path.join(OUT_DIR, "TableS_train_cv_performance.csv"), index=False, encoding="utf-8-sig")
tableS.to_excel(os.path.join(OUT_DIR, "TableS_train_cv_performance.xlsx"), index=False)
print("附表：训练集 10 折交叉验证表现（每名学生的预测来自未见过该学生的模型）")
tableS

In [ ]:
# ===== 6. 图：ROC 曲线（A）与校准曲线（B）。正文用测试集，附图用训练集交叉验证 =====
COLORS = {"LR": "#1f77b4", "RF": "#ff7f0e", "SVM": "#2ca02c", "KNN": "#d62728", "XGB": "#9467bd",
          "MLP": "#8c564b", "LightGBM": "#e377c2", "NB": "#7f7f7f"}

def plot_roc_calibration(y, preds, fname):
    order_ = sorted(MODEL_NAMES, key=lambda m: -roc_auc_score(y, preds[m]))
    plt.rcParams.update(PAPER_RC)
    fig, ax = plt.subplots(1, 2, figsize=(14, 6.2))
    for m in order_:
        fpr, tpr, _ = roc_curve(y, preds[m]); auc, lo, hi = auc_ci_delong(y, preds[m])
        ax[0].plot(fpr, tpr, lw=1.8, color=COLORS[m], label=f"{m} (AUC={auc:.3f}, 95% CI {lo:.3f}-{hi:.3f})")
        fr, mp = calibration_curve(y, preds[m], n_bins=10, strategy="quantile")
        ax[1].plot(mp, fr, lw=1.8, marker="o", ms=3, color=COLORS[m],
                   label=f"{m} (Brier={brier_score_loss(y, preds[m]):.3f})")
    ax[0].plot([0, 1], [0, 1], "k--", lw=1)
    ax[0].set(xlim=(0, 1), ylim=(0, 1), xlabel="1 - Specificity (False Positive Rate)",
              ylabel="Sensitivity (True Positive Rate)")
    ax[0].legend(loc="lower right", fontsize=8.5)
    ax[1].plot([0, 1], [0, 1], "k--", lw=1, label="Perfectly calibrated")
    ax[1].set(xlim=(0, 1), ylim=(0, 1), xlabel="Mean predicted probability", ylabel="Fraction of positives")
    ax[1].legend(loc="upper left", fontsize=8.5)
    for a, lab in zip(ax, "AB"):
        a.text(-0.12, 1.04, lab, transform=a.transAxes, fontsize=20, fontweight="bold")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname + ".png"), bbox_inches="tight")
    plt.savefig(os.path.join(OUT_DIR, fname + ".pdf"), bbox_inches="tight")
    plt.show()
    return order_

order = plot_roc_calibration(y_test, p_test, "Fig_ROC_calibration_test")             # 正文
_ = plot_roc_calibration(y_train, p_train_cv, "FigS_ROC_calibration_train_cv")      # 附图

In [ ]:
# ===== 7. DeLong 检验：各模型与最佳模型比较，以及两两比较的 P 值矩阵 =====
rows = []
for m in MODEL_NAMES:
    if m == BEST_MODEL:
        continue
    d, p = delong_test(y_test, p_test[BEST_MODEL], p_test[m])
    rows.append({"Comparison": f"{BEST_MODEL} vs {m}", "AUC difference": round(d, 4),
                 "DeLong p": f"{p:.3f}" if p >= 0.001 else "<0.001"})
delong_best = pd.DataFrame(rows)
pmat = pd.DataFrame(index=MODEL_NAMES, columns=MODEL_NAMES, dtype=object)
for a in MODEL_NAMES:
    for b in MODEL_NAMES:
        pmat.loc[a, b] = "-" if a == b else f"{delong_test(y_test, p_test[a], p_test[b])[1]:.3f}"
delong_best.to_csv(os.path.join(OUT_DIR, "DeLong_vs_best.csv"), index=False, encoding="utf-8-sig")
pmat.to_csv(os.path.join(OUT_DIR, "DeLong_pairwise_p.csv"), encoding="utf-8-sig")
display(delong_best); display(pmat)

In [ ]:
# ===== 8. 不同假设患病率下最佳模型的 PPV / NPV（用测试集灵敏度、特异度按贝叶斯公式换算）=====
rows = []
t_f1_best = F1_THRESHOLDS[BEST_MODEL]
for label, t in [("Default", 0.50), ("F1-optimal", t_f1_best)]:
    tm = threshold_metrics(y_test, p_test[BEST_MODEL], t); se, sp = tm["Recall"], tm["Specificity"]
    for prev in [0.05, 0.10, 0.20, 0.30]:
        ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
        npv = sp * (1 - prev) / (sp * (1 - prev) + (1 - se) * prev)
        rows.append({"Model": BEST_MODEL, "Decision Label": label, "Threshold": t, "Sensitivity": round(se, 3),
                     "Specificity": round(sp, 3), "Assumed prevalence": prev, "PPV": round(ppv, 3), "NPV": round(npv, 3)})
ppv_sim = pd.DataFrame(rows)
ppv_sim.to_csv(os.path.join(OUT_DIR, "PPV_NPV_by_prevalence.csv"), index=False, encoding="utf-8-sig")
ppv_sim

In [ ]:
# ===== 9. 决策曲线分析（DCA）：最佳模型，标出训练集确定的 F1 最优阈值 =====
def net_benefit(y, p, thresholds):
    n = len(y); out = []
    for t in thresholds:
        pred = p >= t; tp = np.sum(pred & (y == 1)); fp = np.sum(pred & (y == 0))
        out.append(tp / n - fp / n * t / (1 - t))
    return np.array(out)

ths = np.arange(0.01, 1.00, 0.01)
prev = y_test.mean()
nb_model = net_benefit(y_test, p_test[BEST_MODEL], ths)
nb_all = prev - (1 - prev) * ths / (1 - ths)
t_f1 = F1_THRESHOLDS[BEST_MODEL]

with plt.rc_context(PAPER_RC):
    fig, ax = plt.subplots(figsize=(9, 4.2))
    ax.plot(ths, nb_model, color="#1f6fb4", lw=2.6, label=f"{BEST_MODEL} model")
    ax.fill_between(ths, 0, nb_model, where=nb_model > 0, color="#1f6fb4", alpha=0.28, linewidth=0)
    ax.plot(ths, nb_all, color="black", ls="--", lw=1.8, label="Treat all")
    ax.axhline(0, color="gray", ls="--", lw=1.8, label="Treat none")
    ax.axvline(t_f1, color="red", ls=":", lw=2.2, label=f"F1-optimal = {t_f1:.2f}")
    ax.set(xlim=(0, 1), ylim=(-0.05, round(prev + 0.08, 1)), xlabel="Threshold probability", ylabel="Net benefit")
    ax.set_title(f"Decision Curve Analysis for {BEST_MODEL} Model", fontsize=16)
    ax.legend(frameon=False, loc="upper right", fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "Fig_DCA_best_model.png"), bbox_inches="tight")
    plt.savefig(os.path.join(OUT_DIR, "Fig_DCA_best_model.pdf"), bbox_inches="tight")
    plt.show()

dca = pd.DataFrame({"threshold": ths, BEST_MODEL: nb_model, "Treat all": nb_all, "Treat none": 0.0})
for m in MODEL_NAMES:                                   # 其余模型的净获益一并保存，供补充材料
    if m != BEST_MODEL:
        dca[m] = net_benefit(y_test, p_test[m], ths)
dca.round(4).to_csv(os.path.join(OUT_DIR, "DCA_net_benefit.csv"), index=False)

In [ ]:
# ===== 10. SHAP（概率尺度，测试集）：A 平均 |SHAP| 条形图 + 径向图；B 蜂群图 =====
import shap
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

# 种族为字符串变量：SHAP 计算时编码为整数，送入模型前再还原
RACE_LEVELS = CAT_LEVELS["raceeth"]
def to_numeric(df):
    d = df[VARS].copy()
    if "raceeth" in d:
        d["raceeth"] = d["raceeth"].map({k: i for i, k in enumerate(RACE_LEVELS)})
    return d.astype(float)
def model_fn(data):
    d = pd.DataFrame(data, columns=VARS)
    if "raceeth" in d:
        d["raceeth"] = d["raceeth"].round().astype(int).map(dict(enumerate(RACE_LEVELS)))
    return models[BEST_MODEL].predict_proba(d)[:, 1]

Xtr_num, Xte_num = to_numeric(X_train), to_numeric(X_test)
background = Xtr_num.iloc[np.random.RandomState(SEED).choice(len(Xtr_num), size=200, replace=False)]
explainer = shap.Explainer(model_fn, background, feature_names=VARS, seed=SEED)
shap_values = explainer(Xte_num)                       # (n_test, n_features)，预测概率尺度

GROUP_LABELS = {**LABELS, "raceeth": "Race/ethnicity (non-White)", "nicotine_status": "Tobacco/nicotine use",
                "alcohol_status": "Alcohol use", "marijuana_status": "Marijuana use", "sexual_behavior": "Sexual behavior",
                "other_drug_any": "Illicit drug use / Rx opioid misuse"}
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
order = np.argsort(-mean_abs_shap)
feat_sorted = [VARS[i] for i in order]
lab_sorted = [GROUP_LABELS[v] for v in feat_sorted]
shap_sorted = mean_abs_shap[order]
pd.DataFrame({"variable": feat_sorted, "label": lab_sorted, "mean_abs_shap": shap_sorted.round(5),
              "percent": (shap_sorted / shap_sorted.sum() * 100).round(1)}) \
  .to_csv(os.path.join(OUT_DIR, "SHAP_importance_by_variable.csv"), index=False, encoding="utf-8-sig")

cmap = plt.cm.RdBu_r
norm = mcolors.Normalize(vmin=float(shap_sorted.min()), vmax=float(shap_sorted.max()))
colors = [cmap(norm(v)) for v in shap_sorted]

with plt.rc_context(PAPER_RC):
    # ---------- A：条形图 + 径向图 ----------
    fig = plt.figure(figsize=(10, 10), dpi=300)
    left_margin, right_margin, bottom_margin, top_margin = 0.08, 0.04, 0.10, 0.06
    colorbar_w, space = 0.025, 0.04
    plot_bottom, plot_h = bottom_margin, 1.0 - bottom_margin - top_margin
    main_left = left_margin + colorbar_w + space
    main_w = 1.0 - main_left - right_margin
    ax_cbar = fig.add_axes([left_margin, plot_bottom, colorbar_w, plot_h])
    ax_bar = fig.add_axes([main_left, plot_bottom, main_w, plot_h])

    cbar = fig.colorbar(ScalarMappable(cmap=cmap, norm=norm), cax=ax_cbar, orientation="vertical")
    cbar.set_ticks([])
    ax_cbar.text(0.5, 1.01, "High", transform=ax_cbar.transAxes, ha="center", va="bottom", fontsize=13)
    ax_cbar.text(0.5, -0.02, "Low", transform=ax_cbar.transAxes, ha="center", va="top", fontsize=13)
    ax_cbar.text(-1.25, 0.5, "Feature contribution (|SHAP|)", transform=ax_cbar.transAxes,
                 fontsize=14, rotation=90, va="center")

    ypos = np.arange(len(feat_sorted))
    ax_bar.barh(y=ypos, width=shap_sorted, color=colors, height=0.6)
    ax_bar.invert_yaxis()                                  # 重要的在上
    max_w = float(shap_sorted.max())
    ax_bar.set_xlim(max_w * 1.42, 0.0)                     # 0 在右侧，左侧留出标签空间
    ax_bar.set_yticks([])
    label_pad = max_w * 0.025
    for i, (lab, w) in enumerate(zip(lab_sorted, shap_sorted)):
        ax_bar.text(w + label_pad, i, lab, ha="right", va="center", fontsize=12, color="black")   # 柱条左端外侧
    ax_bar.set_xlabel("Mean |SHAP|", fontsize=14)
    ax_bar.spines[["left", "top"]].set_visible(False)
    ax_bar.tick_params(axis="x", which="major", direction="in", length=6, labelsize=12)
    ax_bar.xaxis.set_minor_locator(ticker.AutoMinorLocator(4))
    ax_bar.tick_params(axis="x", which="minor", direction="in", length=4)

    ax_rad = fig.add_axes([main_left - 0.03, plot_bottom + 0.005, 0.36, 0.36], projection="polar")
    ax_rad.patch.set_alpha(0)
    percent = shap_sorted / shap_sorted.sum() * 100
    widths = shap_sorted / shap_sorted.sum() * 2 * np.pi
    n = len(feat_sorted)
    base_len, step, ring_w = 3.0, 0.45, 1.8
    tot_len = [base_len + i * step for i in range(n)]
    inner_h = [max(0, tl - ring_w) for tl in tot_len]
    inner_cols = (["#f7f7f7", "#eeeeee"] * (n // 2 + 1))[:n]
    thetas = np.cumsum([0] + list(widths[:-1])) - np.pi / 21
    ax_rad.bar(thetas, inner_h, width=widths, color=inner_cols, align="edge", edgecolor="white", linewidth=1.1)
    ax_rad.bar(thetas, [ring_w] * n, width=widths, bottom=inner_h, color=colors, align="edge",
               edgecolor="white", linewidth=1.1)
    for i in range(n):
        if percent[i] >= 3:                                # 太窄的扇区不标注，避免重叠
            ax_rad.text(thetas[i] + widths[i] / 2, tot_len[i] + 0.6, f"{percent[i]:.1f}%",
                        ha="center", va="center", fontsize=9)
    ax_rad.set_yticklabels([]); ax_rad.set_xticklabels([])
    ax_rad.spines["polar"].set_visible(False); ax_rad.grid(False)
    ax_rad.set_theta_zero_location("N"); ax_rad.set_theta_direction(-1)
    ax_rad.set_ylim(0, max(tot_len) + 1.5)
    plt.savefig(os.path.join(OUT_DIR, "Fig_SHAP_A_importance.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUT_DIR, "Fig_SHAP_A_importance.pdf"), bbox_inches="tight")
    plt.show()

    # ---------- B：蜂群图（同配色、同顺序）----------
    X_color = Xte_num[feat_sorted].copy()
    if "raceeth" in X_color:                               # 着色：非白人=高，白人=低
        X_color["raceeth"] = (X_color["raceeth"] != 0).astype(int)
    shap.summary_plot(shap_values.values[:, order], X_color, feature_names=lab_sorted, plot_type="dot",
                      show=False, plot_size=(8, 8), cmap=plt.cm.RdBu_r, max_display=len(feat_sorted), sort=False)
    fig = plt.gcf(); fig.set_size_inches(8, 8)
    plt.xlabel("SHAP value (impact on model output)", fontsize=12); plt.ylabel("")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "Fig_SHAP_B_beeswarm.png"), dpi=300, bbox_inches="tight")
    plt.savefig(os.path.join(OUT_DIR, "Fig_SHAP_B_beeswarm.pdf"), bbox_inches="tight")
    plt.show()
print("SHAP 为预测概率尺度；种族着色为非白人（高）/白人（低）")

In [ ]:
# ===== 11. SHAP 个体解释（瀑布图，概率尺度）：测试集中一名未遂与一名未未遂学生 =====
pt_ = p_test[BEST_MODEL]
for lab_, cls in [("attempt", 1), ("no_attempt", 0)]:
    idx = np.where(y_test == cls)[0]
    i = idx[np.argmin(np.abs(pt_[idx] - np.median(pt_[idx])))]
    exp_i = shap.Explanation(values=shap_values.values[i], base_values=shap_values.base_values[i],
                             data=X_test.iloc[i][VARS].values, feature_names=[GROUP_LABELS[v] for v in VARS])
    with plt.rc_context(PAPER_RC):
        shap.plots.waterfall(exp_i, max_display=12, show=False)
        plt.gcf().set_size_inches(12, 7)
        plt.title(f"{BEST_MODEL}: test case with {lab_.replace('_', ' ')} (predicted p = {pt_[i]:.2f})", fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(OUT_DIR, f"Fig_SHAP_waterfall_{lab_}.png"), bbox_inches="tight", dpi=300)
        plt.show()

In [ ]:
# ===== 12. 最佳模型为 LR 时：标准化系数与 OR（惩罚后），并保存全部模型 =====
import joblib
if BEST_MODEL == "LR":
    pre_, clf_ = models["LR"].named_steps["prep"], models["LR"].named_steps["clf"]
    feat_ = [n.split("__", 1)[1] for n in pre_.get_feature_names_out()]
    coef = pd.DataFrame({"feature": [LABELS.get(f, f) for f in feat_], "coef": clf_.coef_[0], "OR": np.exp(clf_.coef_[0])})
    coef.round(4).to_csv(os.path.join(OUT_DIR, "LR_coefficients.csv"), index=False, encoding="utf-8-sig")
    display(coef.round(3))
for m in MODEL_NAMES:
    joblib.dump(models[m], os.path.join(OUT_DIR, f"model_{m}.joblib"))
print("已保存到", os.path.abspath(OUT_DIR)); print(sorted(os.listdir(OUT_DIR)))

In [ ]:
# ===== 14. 研究流程图（Flow chart）=====
# 纳入排除人数来自 CDC 原始数据清洗（见 README），此处直接填写
from matplotlib.patches import FancyBboxPatch
FLOW = {"total": 61722, "age": 6593, "grade": 318, "si": 44123, "sa_missing": 864, "final": 9824, "final_pos": 4027}
n_tr, n_te = len(train), len(test)

def box(ax, x, y, w, h, text, fs=11, dashed=False, align="center", pad=0.012, lw=1.4):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle=f"round,pad={pad},rounding_size=0.012",
                                fc="white", ec="black", lw=lw, ls=(0, (4, 3)) if dashed else "-"))
    if text:
        tx = x + w / 2 if align == "center" else x + 0.012
        ax.text(tx, y + h / 2, text, ha=align, va="center", fontsize=fs, linespacing=1.35)

def arrow(ax, x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", lw=1.4, color="#333333",
                                                                     mutation_scale=14))

with plt.rc_context(PAPER_RC):
    fig, ax = plt.subplots(figsize=(15, 11.5)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    box(ax, 0.30, 0.915, 0.34, 0.06, f"Students in the national YRBS 2019, 2021, 2023 and 2025\n(n = {FLOW['total']:,})", fs=12)
    box(ax, 0.50, 0.775, 0.47, 0.105,
        f"· Age not 14–17 years or missing (n = {FLOW['age']:,})\n"
        f"· Grade not 9–12 or missing (n = {FLOW['grade']:,})\n"
        f"· No serious suicidal ideation in past 12 months or missing (n = {FLOW['si']:,})\n"
        f"· Missing suicide attempt item (n = {FLOW['sa_missing']:,})", fs=11, align="left")
    arrow(ax, 0.47, 0.915, 0.47, 0.745); arrow(ax, 0.50, 0.828, 0.47, 0.828)
    box(ax, 0.27, 0.685, 0.40, 0.06,
        f"Final dataset: adolescents with serious suicidal ideation (n = {FLOW['final']:,})\n"
        f"Suicide attempt n = {FLOW['final_pos']:,} ({FLOW['final_pos'] / FLOW['final']:.1%})", fs=12)
    # 训练/测试
    ax.plot([0.10, 0.84], [0.655, 0.655], color="#333333", lw=1.4); ax.plot([0.47, 0.47], [0.685, 0.655], color="#333333", lw=1.4)
    arrow(ax, 0.10, 0.655, 0.10, 0.615); arrow(ax, 0.84, 0.655, 0.84, 0.615)
    box(ax, 0.02, 0.555, 0.16, 0.06, f"Training dataset\n(70%, n = {n_tr:,})", fs=12)
    box(ax, 0.76, 0.555, 0.16, 0.06, f"Testing dataset\n(30%, n = {n_te:,})", fs=12)
    # 预处理 / 筛选 / 预测变量
    arrow(ax, 0.10, 0.555, 0.10, 0.475)
    box(ax, 0.005, 0.335, 0.20, 0.14, "", dashed=True)
    box(ax, 0.025, 0.415, 0.16, 0.045, "Data preprocessing", fs=12)
    ax.text(0.03, 0.375, "· Composite variable construction\n· RF iterative imputation (fit on training)\n· Standardization\n· One-hot encoding",
            fontsize=9.5, va="center", linespacing=1.35)
    box(ax, 0.235, 0.375, 0.14, 0.10, "", dashed=True)
    box(ax, 0.25, 0.415, 0.11, 0.045, "Feature selection", fs=12)
    ax.text(0.255, 0.39, "· LASSO\n· Boruta", fontsize=10, va="center")
    box(ax, 0.405, 0.375, 0.12, 0.10, "", dashed=True)
    box(ax, 0.415, 0.415, 0.10, 0.045, f"Predictors (n = {len(VARS)})", fs=11)
    ax.text(0.42, 0.392, "· VIF", fontsize=10, va="center")
    arrow(ax, 0.205, 0.437, 0.25, 0.437); arrow(ax, 0.36, 0.437, 0.415, 0.437); arrow(ax, 0.515, 0.437, 0.565, 0.437)
    # 8 种算法
    algos = ["Logistic regression", "Random forest", "Support vector machine", "Extreme gradient boosting",
             "Light GBM", "Multilayer perceptron", "K-nearest neighbors", "Naive bayes"]
    box(ax, 0.565, 0.17, 0.17, 0.48, "", dashed=True)
    for k, a in enumerate(algos):
        box(ax, 0.58, 0.595 - k * 0.052, 0.14, 0.038, a, fs=10.5)
    ax.text(0.572, 0.19, "· Grid search with 10-fold CV\n  for optimal hyperparameters", fontsize=9.5, va="center")
    # 评估
    arrow(ax, 0.84, 0.555, 0.84, 0.13)
    box(ax, 0.77, 0.075, 0.14, 0.055, "Model evaluation\nand comparison", fs=11.5)
    box(ax, 0.36, 0.005, 0.37, 0.155, "", dashed=True)
    box(ax, 0.37, 0.105, 0.35, 0.048, "· AUROC (DeLong test), AUPRC (bootstrap)\n· Accuracy, Recall, Specificity, PPV, NPV, F1", fs=9.5, align="left")
    box(ax, 0.37, 0.058, 0.35, 0.04, "· Calibration curves, H-L test, Brier score", fs=9.5, align="left")
    box(ax, 0.37, 0.013, 0.35, 0.04, "· Decision curve analysis", fs=9.5, align="left")
    for yy, t in [(0.129, "Discrimination"), (0.078, "Calibration"), (0.033, "Clinical utility")]:
        ax.text(0.715, yy, t, ha="right", va="center", fontsize=9.5, style="italic")
    arrow(ax, 0.77, 0.10, 0.73, 0.10)
    arrow(ax, 0.36, 0.08, 0.30, 0.08)
    box(ax, 0.17, 0.055, 0.13, 0.05, "Model explanation\nand application", fs=11)
    arrow(ax, 0.17, 0.08, 0.135, 0.08)
    box(ax, 0.01, 0.03, 0.125, 0.10, "", dashed=True)
    box(ax, 0.02, 0.085, 0.105, 0.035, "SHAP", fs=11)
    box(ax, 0.02, 0.04, 0.105, 0.035, "Online risk calculator", fs=9.5)
    arrow(ax, 0.65, 0.17, 0.65, 0.16)
    plt.savefig(os.path.join(OUT_DIR, "Fig_flowchart.png"), bbox_inches="tight", dpi=300)
    plt.savefig(os.path.join(OUT_DIR, "Fig_flowchart.pdf"), bbox_inches="tight")
    plt.show()